# Verify Behavioral Asymmetry

**Goal:** Test the "Master/Slave" Hypothesis.

**Hypothesis:** The Decentralized Value Asymmetry (Agent 0 ~ 20, Agent 1 ~ -2) is caused by **Behavioral Asymmetry**.
The policy (Sum of V) forces Agent 1 to do all the work (incurring -1 penalties) while Agent 0 does nothing (collecting +2 rewards).

**Metric:** We will run a simulation and count **Apples Picked by Agent 0** vs **Apples Picked by Agent 1**.

In [6]:
# === CONFIGURATION ===
HEIGHT = 6
WIDTH = 6
NUM_AGENTS = 2
SPAWN_PROB = 0.05
DESPAWN_PROB = 0.09
DISCOUNT = 0.99

CNN_CONV_CHANNELS = [32, 64]
CNN_HEAD_HIDDEN_DIM = 128
CNN_HEAD_NUM_LAYERS = 3
CNN_KERNEL_SIZE = 3

SIMULATION_STEPS = 1000

PATH_DECENTRALIZED_AG0 = "decen_cnn_-12_reward/model_agent0_step16050000.pt"
PATH_DECENTRALIZED_AG1 = "decen_cnn_-12_reward/model_agent1_step16050000.pt"

In [7]:
import torch
import numpy as np
from tqdm import tqdm
import sys
import random
sys.path.append("../") 
from models.value_cnn_new import ValueCNNDecentralizedStandard
from tadd_helpers.env_functions import init_empty_state, spawn_apples, despawn_apples, State
from orchard.environment import MoveAction

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
# Load Decentralized Models
dec_models = []
for p in [PATH_DECENTRALIZED_AG0, PATH_DECENTRALIZED_AG1]:
    m = ValueCNNDecentralizedStandard(HEIGHT, WIDTH, 0.0, DISCOUNT, CNN_HEAD_HIDDEN_DIM, CNN_HEAD_NUM_LAYERS, CNN_CONV_CHANNELS, CNN_KERNEL_SIZE).to(DEVICE)
    m.load_state_dict(torch.load(p, map_location=DEVICE))
    m.eval()
    dec_models.append(m)
    
print("Models Loaded.")

Models Loaded.


In [9]:
def get_best_action_decentralized(models, state, agent_idx):
    """
    Selects action that maximizes Sum(V_0 + V_1)
    """
    agent_pos = state.agent_position(agent_idx)
    best_val = -float('inf')
    best_act = None
    
    for action in MoveAction:
        # 1. Simulate Move
        new_pos = np.clip(agent_pos + action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
        s_mid = state.copy()
        s_mid.set_agent_position(agent_idx, new_pos)
        
        # 2. Query Both Models
        team_val = 0.0
        for i in range(NUM_AGENTS):
            # IMPORTANT: Pass the agent's specific position
            pos_i = s_mid.agent_position(i)
            team_val += models[i].get_value(s_mid, agent_pos=pos_i)
            
        if team_val > best_val:
            best_val = team_val
            best_act = action
            
    return best_act

In [10]:
def run_behavior_test():
    state = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
    
    # Statistics
    stats = {
        "picks_0": 0,
        "picks_1": 0,
        "reward_0_cumulative": 0.0,
        "reward_1_cumulative": 0.0
    }
    
    print(f"Running simulation for {SIMULATION_STEPS} steps...")
    
    for _ in tqdm(range(SIMULATION_STEPS)):
        # 1. Pick Active Agent Randomly
        active_idx = state.get_random_agent_id()
        
        # 2. Choose Action using JOINT POLICY
        action = get_best_action_decentralized(dec_models, state, active_idx)
        
        # 3. Execute
        new_pos = np.clip(state.agent_position(active_idx) + action.vector, [0,0], [HEIGHT-1, WIDTH-1])
        state.set_agent_position(active_idx, new_pos)
        
        # 4. Check Apple Pick
        if state.apples[tuple(new_pos)] > 0:
            state.remove_apple_at(new_pos)
            
            # Update Stats
            if active_idx == 0:
                stats["picks_0"] += 1
                stats["reward_0_cumulative"] += -1.0 # Picker penalty
                stats["reward_1_cumulative"] += 2.0  # Observer reward
            else:
                stats["picks_1"] += 1
                stats["reward_1_cumulative"] += -1.0
                stats["reward_0_cumulative"] += 2.0
        
        # 5. Environment Dynamics
        spawn_apples(state, SPAWN_PROB)
        despawn_apples(state, DESPAWN_PROB)
        
    return stats

stats = run_behavior_test()

Running simulation for 1000 steps...


100%|██████████| 1000/1000 [00:28<00:00, 34.67it/s]


In [11]:
# === REPORT ===
total_picks = stats["picks_0"] + stats["picks_1"]
pct_0 = (stats["picks_0"] / total_picks * 100) if total_picks > 0 else 0
pct_1 = (stats["picks_1"] / total_picks * 100) if total_picks > 0 else 0

print(f"\n{'='*30}")
print(f"BEHAVIORAL ANALYSIS REPORT")
print(f"{'='*30}")
print(f"Total Apples Picked: {total_picks}")
print(f"\nWORKLOAD DISTRIBUTION:")
print(f"  Agent 0 Picks: {stats['picks_0']:<5} ({pct_0:.1f}%)")
print(f"  Agent 1 Picks: {stats['picks_1']:<5} ({pct_1:.1f}%)")

print(f"\nREWARD DISTRIBUTION (Simulated):")
print(f"  Agent 0 Net Reward: {stats['reward_0_cumulative']:.1f}")
print(f"  Agent 1 Net Reward: {stats['reward_1_cumulative']:.1f}")

print(f"\nVALUE MODEL CHECK:")
print("  Does this match the Value Predictions? (Ag0 ~ 20, Ag1 ~ -2)")
if stats["reward_0_cumulative"] > stats["reward_1_cumulative"]:
    print("  ✅ YES. Agent 0 is the Net Beneficiary (The 'Master').")
else:
    print("  ❌ NO. Agent 0 is doing the work. The prediction is inverted.")
    
if pct_0 < 40 or pct_1 < 40:
    print("\n⚠️  ROLE COLLAPSE DETECTED: One agent is doing >60% of the work.")
else:
    print("\n✅  NO BEHAVIORAL COLLAPSE: Workload is balanced (50/50).")


BEHAVIORAL ANALYSIS REPORT
Total Apples Picked: 417

WORKLOAD DISTRIBUTION:
  Agent 0 Picks: 117   (28.1%)
  Agent 1 Picks: 300   (71.9%)

REWARD DISTRIBUTION (Simulated):
  Agent 0 Net Reward: 483.0
  Agent 1 Net Reward: -66.0

VALUE MODEL CHECK:
  Does this match the Value Predictions? (Ag0 ~ 20, Ag1 ~ -2)
  ✅ YES. Agent 0 is the Net Beneficiary (The 'Master').

⚠️  ROLE COLLAPSE DETECTED: One agent is doing >60% of the work.
